## Sentiment Analysis Methodoloygy
 
The dataset where the models were tested is available here: https://huggingface.co/datasets/ElKulako/stocktwits-crypto <br>
It contains all cryptocurrency-related posts from the StockTwits website (around 1.3M), from 1st of November 2021 to the 15th of June 2022.
 
The provided data was already cleaned and preprocessed, in particular it was removed:
 
- cashtags, hashtags, usernames,
- URLs, crypto wallets,
- Chinese, Korean and Japanese characters,
- (most) UTF-8 encoding issues
- All posts shorter than 4 words
- All duplicate posts
- Spacing and punctuation issues, converted all text to lowercase

Initially, i intended to test on the dataset the following models:
 
- Vader: Lexicon and rule-based sentiment analysis tool that is specifically attuned to sentiments expressed in social media
- Support Vector Machines: A traditional, statistical classifier.
- Random Forest: An ensemble machine learning method utilizing deep decision trees.
- Gemma 2B: Google's lightweight, 2-billion parameter Large Language Model.

All this models were to be benchmarked against CryptoBERT, a specialized Transformer machine learning model trained on 3.2 millions social media posts of various cryptocurrencies. <br>
The benchmark results for CryptoBERT are publicly available, making it an ideal competitor, especially since it was explicitly fine-tuned on a portion of this exact StockTwits dataset.
 
After some testing i decided to drop Gemma 2B and Random forest for two main reasons:
 
- Performance: Preliminary testing showed they did not maintain competitive accuracy compared to the SVM and CryptoBERT.
- Computational Efficiency: At scale the inference time for the LLM and the training complexity of the Random Forest were prohibitively high.

Consequently, the final evaluation focuses on a comparison between VADER, SVM and CryptoBERT.
 
Results & Evaluation
 
VADER and SVM were evaluated on the same exact 83k rows test set of the CryptoBERT paper
 
VADER: Accuracy: 37.34% | Macro F1: 0.3679
 
As expected, the traditional rule-based lexicon struggled. <br>
VADER's reliance on standard English definitions proved inadequate for capturing the nuanced, slang-heavy and emoji-driven nature of cryptocurrency posts.
 
CryptoBERT: Accuracy: 55.60% | Macro F1: 0.5579
 
The specialized transformer established a balanced standard. <br>
Thanks to its deep contextual embeddings pre-trained on crypto-specific text, it successfully managed the complex class imbalances and set a high benchmark for the Macro F1 score.
 
SVM: Accuracy: 59.71% | Macro F1: 0.5916
 
The SVM performed exceptionally well, outperforming CryptoBERT in both categories. <br>
By utilizing a wide range of n-grams (1–5), the model successfully captured the specific "power phrases" and slang that drive sentiment on StockTwits. <br>
It achieved an accuracy 4.11% higher than the state-of-the-art transformer.
 
Conclusion

The results demonstrate that a properly tuned statistical machine learning pipeline can surpass a specialized Transformer model in both raw accuracy and Macro F1 score on this dataset. <br>
The SVM’s superior performance, combined with its significantly lower computational overhead, makes it the most viable and scalable solution for high-frequency sentiment analysis in the cryptocurrency domain.

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pickle

TARGET_NAMES = ["Bearish", "Neutral", "Bullish"]
LABEL_ORDER  = [0, 1, 2]

vader = SentimentIntensityAnalyzer()

def get_vader_sentiment(text: str) -> int:
    compound = vader.polarity_scores(text)["compound"]
    if compound >= 0.05:
        return 2   # Bullish
    elif compound <= -0.05:
        return 0   # Bearish
    else:
        return 1   # Neutral

def evaluate_model(model_name: str, y_true, y_pred):
    
    print(f" {model_name}")
    print(f"  Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred, labels=LABEL_ORDER, target_names=TARGET_NAMES, digits=4))

if __name__ == "__main__":

    df_train_1 = pd.read_csv("../data/sentiment/stockwits_train_1.csv")
    df_train_2 = pd.read_csv("../data/sentiment/stockwits_train_2.csv")
    df_test = pd.read_csv("../data/sentiment/stockwits_test.csv")

    df_train = pd.concat([df_train_1, df_train_2], ignore_index=True)
    df_tune = df_train.sample(frac=0.2, random_state=42)
    x_tune, y_tune = df_tune["text"].tolist(), df_tune["label"].tolist()

    
    x_train, y_train = df_train["text"].tolist(), df_train["label"].tolist()
    x_test, y_test = df_test["text"].tolist(), df_test["label"].tolist()
    
    results_df = pd.DataFrame({"text": x_test, "label": y_test})

    print("Running VADER...")
    results_df["vader_pred"] = results_df["text"].apply(get_vader_sentiment)
    
    print("Training & running SVM...")
    svm_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LinearSVC(class_weight="balanced", dual=False)),
    ])

    param_grid = {
    "tfidf__ngram_range": [(1,4),(1, 5)], 
    "tfidf__min_df": [1, 2],                       
    "tfidf__max_df": [0.9, 0.95, 1.0],                     
    "tfidf__sublinear_tf": [False],                                
}
    
    print("Running Grid Search (this will take a while)...")
    grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    cv=3,          
    scoring='f1_macro', 
    n_jobs=2,  
    verbose=3       
)
    grid_search.fit(x_tune, y_tune)
    print("\nBest Parameters found:", grid_search.best_params_)
    
    best_svm = grid_search.best_estimator_
    best_svm.fit(x_train, y_train)          
    results_df["svm_pred"] = best_svm.predict(x_test)

    evaluate_model("VADER", results_df["label"], results_df["vader_pred"])
    evaluate_model("SVM", results_df["label"], results_df["svm_pred"])
    
    print(f"CryptoBert")
    print(f"  Accuracy: 0.5560")
    print(f"  Macro AVG F1 Score: 0.5579")
    print(f"  Macro AVG Precision: 0.4658")
    print(f"  Macro AVG Recall: 0.6044")

    pickle.dump(best_svm, open("../data/models/svm.pkl", "wb"))

Running VADER...
Training & running SVM...
Running Grid Search (this will take a while)...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV 1/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.519 total time=  53.7s
[CV 2/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.520 total time=  57.0s
[CV 3/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 4), tfidf__sublinear_tf=False;, score=0.519 total time=  49.7s
[CV 1/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.520 total time= 1.1min
[CV 2/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.521 total time= 1.1min
[CV 3/3] END tfidf__max_df=0.9, tfidf__min_df=1, tfidf__ngram_range=(1, 5), tfidf__sublinear_tf=False;, score=0.519 total time= 1.2min
[CV 1/3] END tfidf__max_df=0.9, tfidf_

## Model Training and Volatility prediction

The goal of this phase is to build a predictive "Early Warning System" for Bitcoin market volatility. <br>
Rather than attempting to predict the direction of the price we are focused on predicting market explosiveness.

Therefore, i define a "Volatility Spike" as an hour where price fluctuations (High-Low range) fall into the top 33.3% of historical moves. <br>
Successfully identifying these regimes allows for advanced risk management, such as tightening stop-losses or exiting leveraged positions before a potential crash.

Our feature set is built on these pillars:

Market Momentum: 1, 3, 6, and 12-hour lags of volatility and returns. This captures the so called "Volatility Clustering", which is the tendency for big moves to follow big moves.
Social Sentiment: We utilize Sentiment Polarity (The "war" between bulls and bears) and the Post Spike Ratio (sudden surges in Reddit chatter volume).
Temporal Context: By including the hour_of_day, we allow the model to learn that volatility behaves differently during the New York Stock Exchange open than it does during other sessions.

Initially, i attempted a Multiclass approach, categorizing the market into three states: Low (0), Neutral (1), and High (2).

While the models successfully identified periods of dead-quiet volatility (Class 0 F1-score: ~0.88) and violent market spikes (Class 2 F1-score: ~0.62), they completely failed to predict the "Neutral" market regime (Class 1 F1-score: ~0.30).

# MULTICLASS CODE

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

def train_and_evaluate(model, x, y, tss, model_name):
    """
    Train and evaluate a classifier using time-series cross-validation.
    Feature importances are averaged across all folds.
    """
    fold = 1
    all_importances = []

    for train_index, test_index in tss.split(x):
        print(f"--- Fold {fold} ---")
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)
        print(classification_report(y_test, y_pred, zero_division=0))

        all_importances.append(model.feature_importances_)
        fold += 1

    mean_importances = np.mean(all_importances, axis=0)
    feature_importance_df = (
        pd.DataFrame({"Feature": x.columns, "Importance": mean_importances})
        .sort_values(by="Importance", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\nTop 15 Features — {model_name} (avg. across folds)")
    print(feature_importance_df.head(15).to_string(index=False)) 

def main():
    df = pd.read_csv("../data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    y = df["BTC_vol_target"].astype(int)

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target"]
    x = df.drop(columns=to_drop_cols)

    tss = TimeSeriesSplit(n_splits=5)

    model_xgb = xgb.XGBClassifier(
        objective="multi:softprob",
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss',
    )
    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        class_weight="balanced",
    )
    model_lgb = lgb.LGBMClassifier(
        objective="multiclass",
        max_depth=3,
        random_state=42,
        class_weight="balanced",
        verbosity=-1,
    )

    print("Training XGBoost Classifier")
    train_and_evaluate(model_xgb, x, y, tss, "XGBoost")

    print("Training Random Forest Classifier")
    train_and_evaluate(model_rf, x, y, tss, "Random Forest")

    print("Training LightGBM Classifier")
    train_and_evaluate(model_lgb, x, y, tss, "LightGBM")

if __name__ == "__main__":
    main()

Training XGBoost Classifier
--- Fold 1 ---
              precision    recall  f1-score   support

           0       0.69      0.51      0.59       650
           1       0.35      0.40      0.38       472
           2       0.35      0.46      0.40       334

    accuracy                           0.47      1456
   macro avg       0.47      0.46      0.46      1456
weighted avg       0.50      0.47      0.48      1456

--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.55      0.77      0.64       371
           1       0.44      0.31      0.37       408
           2       0.73      0.70      0.71       677

    accuracy                           0.61      1456
   macro avg       0.57      0.59      0.57      1456
weighted avg       0.60      0.61      0.60      1456

--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.51      0.52      0.52       475
           1       0.41      0.40      0.40       540
  

The Solution: The 66/33 Split <br><br>
To maximize the predictive power of our sentiment features and align with our "alarm" objective, we changed the architecture from Multiclass to Binary. <br>
Instead of forcing the model to guess three states, we merged the bottom 66% of market volatility into a single baseline class, isolating the top 33% as our target:

Class 0 ("Normal Market"): The bottom 66.6% of volatility (Low + Neutral). <br>
Class 1 ("Volatility Spike"): The top 33.3% of volatility (High).

By removing the noisy boundary between Low and Neutral, we allow the models to focus strictly on the extreme outliers, the events that actually trigger liquidations.

# BINARY CODE

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report

def train_and_evaluate(model, x, y, tss, model_name):
    """
    Train and evaluate a classifier using time-series cross-validation.
    Feature importances are averaged across all folds.
    """
    fold = 1
    all_importances = []

    for train_index, test_index in tss.split(x):
        print(f"--- Fold {fold} ---")
        x_train, x_test = x.iloc[train_index], x.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        model.fit(x_train, y_train)

        y_pred = model.predict(x_test)
        print(classification_report(y_test, y_pred, zero_division=0))

        all_importances.append(model.feature_importances_)
        fold += 1

    mean_importances = np.mean(all_importances, axis=0)
    feature_importance_df = (
        pd.DataFrame({"Feature": x.columns, "Importance": mean_importances})
        .sort_values(by="Importance", ascending=False)
        .reset_index(drop=True)
    )

    print(f"\nTop 15 Features — {model_name} (avg. across folds)")
    print(feature_importance_df.head(15).to_string(index=False)) 

def main():
    df = pd.read_csv("../data/clean_reddit/final_submission_clean.csv")
    df.set_index("index", inplace=True)
    
    df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)
    y = df["BTC_vol_target_binary"]

    to_drop_cols = ["BTC_vol_target", "ETH_vol_target", "SOL_vol_target", "BTC_vol_target_binary"]
    x = df.drop(columns=to_drop_cols)

    tss = TimeSeriesSplit(n_splits=5, gap=24)

    model_xgb = xgb.XGBClassifier(
        objective="binary:logistic",
        max_depth=5,
        learning_rate=0.05,
        random_state=42,
        eval_metric='logloss',
    )
    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
        class_weight="balanced",
    )
    model_lgb = lgb.LGBMClassifier(
        objective="binary",
        max_depth=3,
        learning_rate=0.05,
        random_state=42,
        class_weight="balanced",
        verbosity=-1,
    )

    print("Training XGBoost Classifier")
    train_and_evaluate(model_xgb, x, y, tss, "XGBoost")

    print("Training Random Forest Classifier")
    train_and_evaluate(model_rf, x, y, tss, "Random Forest")

    print("Training LightGBM Classifier")
    train_and_evaluate(model_lgb, x, y, tss, "LightGBM")

if __name__ == "__main__":
    main()

/tmp/ipykernel_27913/104008356.py:44: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["BTC_vol_target_binary"] = df["BTC_vol_target"].apply(lambda x: 1 if int(x) == 2 else 0)


Training XGBoost Classifier
--- Fold 1 ---
              precision    recall  f1-score   support

           0       0.82      0.85      0.83      1122
           1       0.42      0.35      0.38       334

    accuracy                           0.74      1456
   macro avg       0.62      0.60      0.61      1456
weighted avg       0.72      0.74      0.73      1456

--- Fold 2 ---
              precision    recall  f1-score   support

           0       0.71      0.86      0.78       779
           1       0.79      0.60      0.68       677

    accuracy                           0.74      1456
   macro avg       0.75      0.73      0.73      1456
weighted avg       0.75      0.74      0.73      1456

--- Fold 3 ---
              precision    recall  f1-score   support

           0       0.76      0.86      0.81      1015
           1       0.54      0.38      0.44       441

    accuracy                           0.71      1456
   macro avg       0.65      0.62      0.62      1456
w